In [2]:
import numpy as np
import pickle
import os
import time


# ===========================================================
# =============== MoSS MULTICLASSE (DIRICHLET) ==============
# ===========================================================

def moss_multiclass_dirichlet(
    n_samples: int,
    alpha: np.ndarray,
    merge: float,
    eps: float = 1e-3,
):
    """
    Gera uma curva MoSS multiclasse via Dirichlet controlada.
    """
    n_classes = len(alpha)
    merge = np.clip(merge, 0.0, 1.0)

    # escala global de concentração (controla separabilidade)
    scale = 50 * (1 - merge) + 5

    scores = np.zeros((n_samples, n_classes))

    # número de amostras por classe
    n_per_class = np.floor(n_samples * alpha).astype(int)
    n_per_class[-1] = n_samples - n_per_class[:-1].sum()

    idx = 0
    for c in range(n_classes):

        # centro suavizado no simplex
        center = np.full(n_classes, 1.0 / n_classes)
        center[c] = 1.0

        center = center / center.sum()

        # mistura controlada + epsilon estrutural
        conc = eps + scale * (
            (1 - merge) * center + merge * np.full(n_classes, 1.0 / n_classes)
        )

        samples = np.random.dirichlet(conc, size=n_per_class[c])
        scores[idx:idx + n_per_class[c]] = samples
        idx += n_per_class[c]

    np.random.shuffle(scores)
    return scores


# ===========================================================
# ========== GERADOR DE DISTRIBUIÇÕES MOSS MULTI ============
# ===========================================================

def gerar_distribuicoes_moss_multiclasse(
    n_samples,
    n_classes,
    n_prevalences,
    n_merges,
    n_curves,
    save_path
):
    """
    Gera um arquivo MoSS multiclasse de grande porte,
    com complexidade comparável ao MoSS binário gigante.
    """

    # Prevalências no simplex
    prevalences = np.random.dirichlet(
        alpha=np.ones(n_classes),
        size=n_prevalences
    )

    merges = np.linspace(0.0, 1.0, n_merges)

    synthetic_distributions = {}
    total = len(prevalences) * len(merges)
    count = 0

    print(f"\n🚀 Gerando MoSS MULTICLASSE: {save_path}")
    print(f"Configuração:")
    print(f"  n_samples     = {n_samples:,}")
    print(f"  n_classes     = {n_classes}")
    print(f"  n_prevalences = {n_prevalences}")
    print(f"  n_merges      = {n_merges}")
    print(f"  n_curves      = {n_curves}")
    print(f"  Total blocos  = {total}\n")

    start_global = time.perf_counter()
    last_times = []

    for alpha in prevalences:
        alpha_key = tuple(np.round(alpha, 4))

        for merge in merges:
            start = time.perf_counter()

            curves = []
            for _ in range(n_curves):
                scores = moss_multiclass_dirichlet(
                    n_samples=n_samples,
                    alpha=alpha,
                    merge=merge
                )
                curves.append(scores)

            synthetic_distributions[(alpha_key, round(merge, 4))] = curves

            block_time = time.perf_counter() - start
            last_times.append(block_time)
            count += 1

            if count % 5 == 0 or count == total:
                avg = np.mean(last_times[-10:])
                remaining = (total - count) * avg
                print(
                    f"[{count}/{total}] "
                    f"Bloco: {block_time:6.2f}s | "
                    f"Média: {avg:6.2f}s | "
                    f"Restante: {remaining/60:6.1f} min"
                )

    with open(save_path, "wb") as f:
        pickle.dump(synthetic_distributions, f, protocol=pickle.HIGHEST_PROTOCOL)

    total_runtime = time.perf_counter() - start_global

    print("\n✔ Finalizado")
    print(f"Arquivo: {save_path}")
    print(f"⏱ Tempo total: {total_runtime/60:,.2f} minutos\n")

    return synthetic_distributions


# ===========================================================
# ===================== CONFIGURAÇÕES =======================
# ===========================================================

MASSIVE_CONFIG_MULTI = dict(
    n_samples=10_000,
    n_classes=4,
    n_prevalences=40,
    n_merges=40,
    n_curves=100,
    save_path="moss_outputs/moss_GIGANTE_MULTICLASSE.pkl"
)


# ===========================================================
# ======================= EXECUÇÃO ==========================
# ===========================================================

if __name__ == "__main__":
    os.makedirs("moss_outputs", exist_ok=True)

    gerar_distribuicoes_moss_multiclasse(**MASSIVE_CONFIG_MULTI)

    print("🏁 MoSS multiclasse gigante gerado com sucesso.")


🚀 Gerando MoSS MULTICLASSE: moss_outputs/moss_GIGANTE_MULTICLASSE.pkl
Configuração:
  n_samples     = 10,000
  n_classes     = 4
  n_prevalences = 40
  n_merges      = 40
  n_curves      = 100
  Total blocos  = 1600

[5/1600] Bloco:   0.45s | Média:   0.45s | Restante:   12.0 min
[10/1600] Bloco:   0.45s | Média:   0.45s | Restante:   12.0 min
[15/1600] Bloco:   0.45s | Média:   0.45s | Restante:   11.9 min
[20/1600] Bloco:   0.45s | Média:   0.45s | Restante:   11.9 min
[25/1600] Bloco:   0.45s | Média:   0.45s | Restante:   11.9 min
[30/1600] Bloco:   0.45s | Média:   0.45s | Restante:   11.9 min
[35/1600] Bloco:   0.45s | Média:   0.45s | Restante:   11.8 min
[40/1600] Bloco:   0.45s | Média:   0.45s | Restante:   11.8 min
[45/1600] Bloco:   0.45s | Média:   0.45s | Restante:   11.7 min
[50/1600] Bloco:   0.45s | Média:   0.45s | Restante:   11.7 min
[55/1600] Bloco:   0.45s | Média:   0.45s | Restante:   11.6 min
[60/1600] Bloco:   0.45s | Média:   0.45s | Restante:   11.6 min
[65